<a href="https://colab.research.google.com/github/heyanugrah/CNNronaldoMessiClassification/blob/main/mytransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Basic Transformer- Encoder

In [1]:
import torch
import torch.nn as nn
import math

In [2]:
# 1. Define the Vocabulary (Adding '[sep]')
new_vocab = {'<pad>': 0, '<unk>': 1, '<cls>': 2, '[sep]': 3, 'this': 4, 'is': 5, 'a': 6, 'positive': 7, 'sentence': 8, 'learning': 9, 'about': 10, 'transformers': 11, 'fun': 12, 'i': 13, 'hate': 14, 'am': 15, 'so': 16, 'disappointed': 17, 'really': 18, 'enjoyed': 19, 'movie': 20, 'wonderful': 21, 'day': 22, 'for': 23, 'walk': 24, 'product': 25, 'excellent!': 26, 'absolutely': 27, 'dreadful': 28, 'service': 29, 'what': 30, 'terrible': 31, 'experience': 32, 'feeling': 33, 'very': 34, 'happy': 35, 'today': 36, 'boy': 37, 'anugrah': 38}

# 2. Define the Reverse Vocabulary
idx_to_word = {i: word for word, i in new_vocab.items()}

vocab_size = len(new_vocab)

print(f"Vocabulary size is: {vocab_size}")

Vocabulary size is: 39


### Modular Tokenization Functions

To improve clarity and reusability, let's break down the tokenization process into several distinct functions:

1.  `_add_special_tokens_and_segments`: Adds `<cls>` and `[sep]` tokens and initializes segment IDs.
2.  `_convert_to_ids`: Converts token strings to their numerical IDs.
3.  `_pad_ids_and_segments`: Handles padding for both token IDs and segment IDs.
4.  `_create_attention_mask_from_ids`: Generates the attention mask from padded token IDs.
5.  `prepare_transformer_inputs`: An orchestrating function that calls the above helpers to produce all necessary inputs for a Transformer model.

In [3]:
# Helper to add special tokens and initial segment IDs
def _add_special_tokens_and_segments(input_text, vocab):
    tokens = [vocab.get("<cls>", "<cls>")] # Start with <cls>
    segment_ids = [0] # <cls> belongs to segment 0

    # Tokenize input text and add words
    for word in input_text.lower().replace('.', '').split():
        tokens.append(word)
        segment_ids.append(0) # All words in the first sentence belong to segment 0

    tokens.append(vocab.get("[sep]", "[sep]")) # Add [sep]
    segment_ids.append(0) # [sep] belongs to segment 0

    return tokens, segment_ids

# Helper to convert tokens to IDs
def _convert_to_ids(tokens, vocab):
    return [vocab.get(token, vocab["<unk>"]) for token in tokens]

# Helper to pad token IDs and segment IDs
def _pad_ids_and_segments(token_ids, segment_ids, max_seq_len, pad_token_id):
    if len(token_ids) > max_seq_len:
        token_ids = token_ids[:max_seq_len]
        segment_ids = segment_ids[:max_seq_len]
    else:
        padding_length = max_seq_len - len(token_ids)
        token_ids.extend([pad_token_id] * padding_length)
        segment_ids.extend([0] * padding_length) # Padding tokens belong to segment 0

    return token_ids, segment_ids

# Helper to create attention mask
def _create_attention_mask_from_ids(token_ids, pad_token_id):
    mask = [1 if token_id != pad_token_id else 0 for token_id in token_ids]
    return torch.tensor(mask, dtype=torch.long).unsqueeze(0).unsqueeze(1).unsqueeze(2) # [batch_size, 1, 1, seq_len]

In [4]:
def prepare_transformer_inputs(input_text, vocab, max_seq_len):
    pad_token_id = vocab['<pad>']

    # Step 1: Add special tokens and assign initial segment IDs
    tokens, segment_ids = _add_special_tokens_and_segments(input_text, vocab)

    # Step 2: Convert tokens to IDs
    token_ids = _convert_to_ids(tokens, vocab)

    # Step 3: Pad token IDs and segment IDs
    padded_token_ids, padded_segment_ids = _pad_ids_and_segments(
        token_ids, segment_ids, max_seq_len, pad_token_id
    )

    # Convert to PyTorch tensors and add batch dimension
    input_ids = torch.tensor(padded_token_ids, dtype=torch.long).unsqueeze(0)
    token_type_ids = torch.tensor(padded_segment_ids, dtype=torch.long).unsqueeze(0)

    # Step 4: Create attention mask
    attention_mask = _create_attention_mask_from_ids(padded_token_ids, pad_token_id)

    return input_ids, attention_mask, token_type_ids

### Demonstration of the new tokenization pipeline

In [5]:


input_text = "anugrah is learning"
max_seq_len = 10 # maximum token need to be created

# Prepare inputs using the new modular function
input_ids, attention_mask_new, token_type_ids_new = prepare_transformer_inputs(
    input_text, new_vocab, max_seq_len
)

print(f"Input Text: '{input_text}'")
print(f"Input IDs: {input_ids}")
print(f"Attention Mask: {attention_mask_new}")
print(f"Token Type IDs: {token_type_ids_new}")

print(f"Input IDs shape: {input_ids.shape}")
print(f"Attention Mask shape: {attention_mask_new.shape}")
print(f"Token Type IDs shape: {token_type_ids_new.shape}")

Input Text: 'anugrah is learning'
Input IDs: tensor([[ 1, 38,  5,  9,  1,  0,  0,  0,  0,  0]])
Attention Mask: tensor([[[[1, 1, 1, 1, 1, 0, 0, 0, 0, 0]]]])
Token Type IDs: tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
Input IDs shape: torch.Size([1, 10])
Attention Mask shape: torch.Size([1, 1, 1, 10])
Token Type IDs shape: torch.Size([1, 10])


### Create and apply embedding

In [6]:
d_model = 8 #vector dimension for each word

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=d_model
)

embedded_input = embedding(input_ids)
print(f"Shape of embedded input: {embedded_input.shape}")

Shape of embedded input: torch.Size([1, 10, 8])


In [7]:
print(embedded_input)

tensor([[[-1.2921, -0.6676,  2.2493, -1.6024,  0.4282,  0.4261, -0.2642,
           1.1694],
         [ 0.1978, -0.6129,  0.2346, -0.5724, -1.0971, -0.5445,  0.2749,
           0.3146],
         [-0.4911, -0.3515,  0.4811, -2.0335,  0.1261,  0.5347,  0.1880,
          -0.3976],
         [ 1.5695,  0.5724, -0.0846,  1.6519, -1.6244,  0.4524,  0.1903,
           0.9587],
         [-1.2921, -0.6676,  2.2493, -1.6024,  0.4282,  0.4261, -0.2642,
           1.1694],
         [ 2.3261, -0.0168,  0.2738,  0.9514,  0.2579,  0.0422, -0.6372,
          -0.7278],
         [ 2.3261, -0.0168,  0.2738,  0.9514,  0.2579,  0.0422, -0.6372,
          -0.7278],
         [ 2.3261, -0.0168,  0.2738,  0.9514,  0.2579,  0.0422, -0.6372,
          -0.7278],
         [ 2.3261, -0.0168,  0.2738,  0.9514,  0.2579,  0.0422, -0.6372,
          -0.7278],
         [ 2.3261, -0.0168,  0.2738,  0.9514,  0.2579,  0.0422, -0.6372,
          -0.7278]]], grad_fn=<EmbeddingBackward0>)


In [8]:
print(f"Input Text: '{input_text}'")
print(f"Input IDs: {input_ids.squeeze(0)}")

print("\nEncodings for each token in 'anugrah is learning':")
# Iterate through the input_ids and corresponding embedded vectors
for i, token_id in enumerate(input_ids.squeeze(0)):
    word = idx_to_word.get(token_id.item(), '<unknown>')
    embedding_vector = embedded_input.squeeze(0)[i]
    print(f"Token: '{word}' (ID: {token_id.item()}) -> Embedding: {embedding_vector.tolist()}")

Input Text: 'anugrah is learning'
Input IDs: tensor([ 1, 38,  5,  9,  1,  0,  0,  0,  0,  0])

Encodings for each token in 'anugrah is learning':
Token: '<unk>' (ID: 1) -> Embedding: [-1.2921457290649414, -0.6676036715507507, 2.249321222305298, -1.602356195449829, 0.4282497465610504, 0.42606520652770996, -0.26420387625694275, 1.1694432497024536]
Token: 'anugrah' (ID: 38) -> Embedding: [0.19776608049869537, -0.6129346489906311, 0.2345838099718094, -0.5723757147789001, -1.0971076488494873, -0.5444581508636475, 0.27486127614974976, 0.31457093358039856]
Token: 'is' (ID: 5) -> Embedding: [-0.49111345410346985, -0.35147443413734436, 0.48106849193573, -2.03346848487854, 0.12611344456672668, 0.5347089171409607, 0.18797175586223602, -0.39757031202316284]
Token: 'learning' (ID: 9) -> Embedding: [1.569486379623413, 0.5723974108695984, -0.08461281657218933, 1.6518632173538208, -1.6243805885314941, 0.4523879289627075, 0.19032153487205505, 0.9586741328239441]
Token: '<unk>' (ID: 1) -> Embedding: [-1

In [9]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=5000):
        super().__init__()

        self.dropout = nn.Dropout(0.1)

        # Create a positional encoding matrix
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term) if d_model % 2 == 0 else torch.cos(position * div_term[:-1])

        pe = pe.unsqueeze(0) # Add batch dimension
        self.register_buffer('pe', pe)

    def forward(self, x):
        # Add positional encoding to the input embeddings
        # x has shape (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

# Instantiate Positional Encoding
pos_encoder = PositionalEncoding(d_model, max_seq_len)

# Apply positional encoding to the embedded input
embedded_with_pos = pos_encoder(embedded_input)

print(f"Shape of embedded input with positional encoding: {embedded_with_pos.shape}")

Shape of embedded input with positional encoding: torch.Size([1, 10, 8])


In [10]:
print(embedded_with_pos[0][1],embedded_with_pos.shape) #PE for anugrah (1,batch,10 total , 8 dimension)

tensor([ 1.1547, -0.0807,  0.3716,  0.4696, -1.2079,  0.5061,  0.0000,  1.4606],
       grad_fn=<SelectBackward0>) torch.Size([1, 10, 8])


In [11]:
print(f"Input Text: '{input_text}'")
print(f"Input IDs: {input_ids.squeeze(0)}")

print("\nCombined Embeddings (Word Embedding + Positional Encoding) for each token:")
for i, token_id in enumerate(input_ids.squeeze(0)):
    word = idx_to_word.get(token_id.item(), '<unknown>')
    combined_embedding_vector = embedded_with_pos.squeeze(0)[i]
    print(f"Token: '{word}' (ID: {token_id.item()}) -> Combined Embedding: {combined_embedding_vector.tolist()}")

Input Text: 'anugrah is learning'
Input IDs: tensor([ 1, 38,  5,  9,  1,  0,  0,  0,  0,  0])

Combined Embeddings (Word Embedding + Positional Encoding) for each token:
Token: '<unk>' (ID: 1) -> Combined Embedding: [-1.4357175827026367, 0.3693292737007141, 0.0, -0.6692847013473511, 0.0, 1.584517002105713, -0.29355987906455994, 0.0]
Token: 'anugrah' (ID: 38) -> Combined Embedding: [1.154707908630371, -0.08070257306098938, 0.3715746998786926, 0.46958720684051514, -1.2078975439071655, 0.5061020851135254, 0.0, 1.4606338739395142]
Token: 'is' (ID: 5) -> Combined Embedding: [0.46464887261390686, -0.8529125452041626, 0.7552642822265625, -1.1704466342926025, 0.16234679520130157, 1.7050100564956665, 0.21107974648475647, 0.6693641543388367]
Token: 'learning' (ID: 9) -> Combined Embedding: [1.900673747062683, -0.463994562625885, 0.23434153199195862, 2.8968887329101562, -1.7715390920639038, 1.6132644414901733, 0.21480169892311096, 0.0]
Token: '<unk>' (ID: 1) -> Combined Embedding: [-2.27660942077

In [12]:
token_type_embedding = nn.Embedding(2, d_model) # Assuming 2 types: 0 for sentence A, 1 for sentence B

'''
Segment Embedding

Tells the model which sentence a token belongs to.

Example:

Sentence A: I love AI
Sentence B: It is powerful

BERT input:

[CLS] I love AI [SEP] It is powerful [SEP]
'''

# Get segment embeddings for our token_type_ids
segment_embeddings = token_type_embedding(token_type_ids_new)

# Add the segment embeddings to the combined word and positional embeddings
final_transformer_input = embedded_with_pos + segment_embeddings

print(f"Shape of token_type_ids: {token_type_ids_new.shape}")
print(f"Shape of segment embeddings: {segment_embeddings.shape}")
print(f"Shape of final input to Transformer: {final_transformer_input.shape}")

print("\nSegment Embeddings:")
print(segment_embeddings)

Shape of token_type_ids: torch.Size([1, 10])
Shape of segment embeddings: torch.Size([1, 10, 8])
Shape of final input to Transformer: torch.Size([1, 10, 8])

Segment Embeddings:
tensor([[[ 0.5593,  1.7058, -1.1002, -1.1989, -1.2870,  0.8005, -1.1062,
           0.1169],
         [ 0.5593,  1.7058, -1.1002, -1.1989, -1.2870,  0.8005, -1.1062,
           0.1169],
         [ 0.5593,  1.7058, -1.1002, -1.1989, -1.2870,  0.8005, -1.1062,
           0.1169],
         [ 0.5593,  1.7058, -1.1002, -1.1989, -1.2870,  0.8005, -1.1062,
           0.1169],
         [ 0.5593,  1.7058, -1.1002, -1.1989, -1.2870,  0.8005, -1.1062,
           0.1169],
         [ 0.5593,  1.7058, -1.1002, -1.1989, -1.2870,  0.8005, -1.1062,
           0.1169],
         [ 0.5593,  1.7058, -1.1002, -1.1989, -1.2870,  0.8005, -1.1062,
           0.1169],
         [ 0.5593,  1.7058, -1.1002, -1.1989, -1.2870,  0.8005, -1.1062,
           0.1169],
         [ 0.5593,  1.7058, -1.1002, -1.1989, -1.2870,  0.8005, -1.1062,
     

In [13]:
print('final transformer input')
print(final_transformer_input)

final transformer input
tensor([[[-8.7637e-01,  2.0751e+00, -1.1002e+00, -1.8682e+00, -1.2870e+00,
           2.3850e+00, -1.3998e+00,  1.1692e-01],
         [ 1.7141e+00,  1.6251e+00, -7.2865e-01, -7.2931e-01, -2.4949e+00,
           1.3066e+00, -1.1062e+00,  1.5776e+00],
         [ 1.0240e+00,  8.5291e-01, -3.4496e-01, -2.3693e+00, -1.1247e+00,
           2.5055e+00, -8.9514e-01,  7.8629e-01],
         [ 2.4600e+00,  1.2418e+00, -8.6588e-01,  1.6980e+00, -3.0586e+00,
           2.4137e+00, -8.9142e-01,  1.1692e-01],
         [-1.7173e+00,  2.3777e-01,  1.8317e+00, -1.9559e+00, -7.6675e-01,
           2.3841e+00, -1.3953e+00,  2.5274e+00],
         [ 2.0785e+00,  2.0023e+00, -2.6328e-01,  8.3331e-01, -9.4488e-01,
           1.9571e+00, -1.8086e+00,  4.1935e-01],
         [ 2.8335e+00,  2.7540e+00, -1.6860e-01,  7.7525e-01, -1.2870e+00,
           1.9565e+00, -1.8075e+00,  4.1934e-01],
         [ 3.8739e+00,  1.7058e+00, -8.0179e-02,  7.0804e-01, -9.2270e-01,
           1.9558e+00, -1.

In [14]:
import torch
import torch.nn as nn
import math

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ffn_hidden_dim, dropout_rate=0.1):
        super().__init__()

        '''
          d_model = 512
          num_heads = 8

          head_dim = 64
          512 dimensions
          ├─ Head 1 : 64
          ├─ Head 2 : 64
          ├─ Head 3 : 64
          ├─ Head 4 : 64
          ├─ Head 5 : 64
          ├─ Head 6 : 64
          ├─ Head 7 : 64
          └─ Head 8 : 64
        '''

        assert d_model % num_heads == 0 #if it is divisible then return true and allow run

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Multi-Head Attention
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

        self.attn_dropout = nn.Dropout(dropout_rate)

        # LayerNorm after MHA residual
        self.layer_norm1 = nn.LayerNorm(d_model)

        # Feed Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_hidden_dim),
            nn.ReLU(),
            nn.Linear(ffn_hidden_dim, d_model)
        )

        self.ffn_dropout = nn.Dropout(dropout_rate)

        # LayerNorm after FFN residual
        self.layer_norm2 = nn.LayerNorm(d_model)

    def _calculate_attention(self, Q, K, V, mask=None):

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        ) / math.sqrt(self.head_dim) # The division by sqrt(head_dim) is the scaling factor.

        if mask is not None:
            # Apply mask to the scores. Masked positions should be set to -inf.
            # This code is used to hide certain tokens from attention.1allowed 0 not allowed
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        output = torch.matmul(
            attention_weights,
            V
        )

        return output

    def forward(self, x, mask=None):
        # In your current setup (cell cbe1d5c1), `mask` is indeed passed as a tensor
        # (attention_mask_new), not None, when calling this layer.

        batch_size, seq_len, _ = x.shape

        # ----------------------------
        # Multi-Head Self Attention
        # ----------------------------

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        '''
        Full token embedding (8 dims)
                  │
                  ▼
            Split into 2 heads
                  │
          ┌──────┴──────┐
          ▼             ▼
        Head 1        Head 2
        (4 dims)      (4 dims)
          │             │
        Attention    Attention
        separately   separately
        '''

        '''
        # Split Q into multiple attention heads and rearrange dimensions
        # From: [batch_size, seq_len, d_model]
        # To:   [batch_size, num_heads, seq_len, head_dim]
        '''
        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        attention_output = self._calculate_attention(
            Q,
            K,
            V,
            mask=mask
        )
        '''
        Before attention:

        Token
        [1,2,3,4,5,6,7,8]

        ↓ split

        Head1 [1,2,3,4]
        Head2 [5,6,7,8]

        After attention:
        Head1 [1,2,3,4]
        Head2 [5,6,7,8]

        ↓ concatenate
        combined attention heads : [1,2,3,4,5,6,7,8]
        '''
        attention_output = (
            attention_output
            .transpose(1, 2)
            .contiguous()
            .view(
                batch_size,
                seq_len,
                self.d_model
            )
        )
        '''
        Multi-head outputs
              ↓
        Concatenate
              ↓
        Wo (Linear Layer)
              ↓
        Final attention representation

        [1,2,3,4,5,6,7,8]
            ↓
        Linear Wo
            ↓
       [5,-2,8,1,4,9,3,7]
        '''

        mha_output = self.Wo(
            attention_output
        )
        '''
        Some output values are randomly set to 0.Prevent Overfitting

        Before:
        [5,-2,8,1,4,9,3,7]

        After dropout:
        [5,0,8,1,0,9,3,7]
        '''
        final_mha_output = self.attn_dropout(
            mha_output
        )

        '''
          Input
            ↓
          Wq, Wk, Wv
            ↓
          Split into heads
            ↓
          Attention
            ↓
          Merge heads
            ↓
          Wo
            ↓
          Dropout
            ↓
          Final MHA output
        '''

        # Residual Connection + LayerNorm
        x = self.layer_norm1(
            x + final_mha_output
        )

        # ----------------------------
        # Feed Forward Network
        # ----------------------------

        ffn_output = self.ffn(x)

        ffn_output = self.ffn_dropout(
            ffn_output
        )

        # Residual Connection + LayerNorm
        output = self.layer_norm2(
            x + ffn_output
        )

        return output

# --- Execution Code ---
encoder_layer = TransformerEncoderLayer(d_model=8, num_heads=2, ffn_hidden_dim=32)
encoder_output = encoder_layer(final_transformer_input, mask=attention_mask_new)

print(f"Encoder Layer Input Shape: {final_transformer_input.shape}")
print(f"Encoder Layer Output Shape: {encoder_output.shape}")
print("\nFirst few values of the output:")
print(encoder_output[0, 0, :])


Encoder Layer Input Shape: torch.Size([1, 10, 8])
Encoder Layer Output Shape: torch.Size([1, 10, 8])

First few values of the output:
tensor([-0.8414,  1.4327, -0.7409, -0.2446, -0.2947,  1.9396, -0.6229, -0.6276],
       grad_fn=<SelectBackward0>)


In [15]:

# --- Execution Code ---
encoder_layer = TransformerEncoderLayer(d_model=8, num_heads=2, ffn_hidden_dim=32)
encoder_output = encoder_layer(final_transformer_input, mask=attention_mask_new)

print(f"Encoder Layer Input Shape: {final_transformer_input.shape}")
print(f"Encoder Layer Output Shape: {encoder_output.shape}")
print("\nFirst few values of the output:")
print(encoder_output[0, 0, :])

Encoder Layer Input Shape: torch.Size([1, 10, 8])
Encoder Layer Output Shape: torch.Size([1, 10, 8])

First few values of the output:
tensor([-0.2606,  1.9295, -1.1920, -0.7929, -0.5378,  1.2516, -0.5339,  0.1361],
       grad_fn=<SelectBackward0>)


In [19]:
targets = torch.cat([input_ids[:, 1:], torch.tensor([[0]])], dim=1)

# Add a linear projection layer from d_model to vocab_size
# d_model is 8, vocab_size is 39
projection_layer = nn.Linear(d_model, vocab_size)

# 2. Define Loss and Optimizer
criterion = nn.CrossEntropyLoss(ignore_index=0)
# Update optimizer to include parameters of the new projection_layer
optimizer = torch.optim.Adam(list(encoder_layer.parameters()) + list(projection_layer.parameters()), lr=0.01)

# DETACH input to prevent backwarding through the previous embedding/PE graph
train_input = final_transformer_input.detach()

# 3. Training Loop
encoder_layer.train()
projection_layer.train() # Set projection layer to training mode
for epoch in range(100):
    optimizer.zero_grad()

    # Forward pass through encoder
    encoder_output = encoder_layer(train_input, mask=attention_mask_new)
    # Apply the projection layer to get logits
    logits = projection_layer(encoder_output)

    # Reshape for CrossEntropyLoss
    loss = criterion(logits.view(-1, vocab_size), targets.view(-1))

    # Backward pass and optimize
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/100], Loss: {loss.item():.4f}")

print("\nTraining complete!")
encoder_layer.eval()
projection_layer.eval() # Set projection layer to evaluation mode
with torch.no_grad():
    new_encoder_output = encoder_layer(train_input, mask=attention_mask_new)
    new_logits = projection_layer(new_encoder_output) # Apply projection
    # For prediction, we are interested in the token at index 1 (after '<cls>') for its prediction.
    # The 'anugrah' token is at index 1 in the input_ids (after '<cls>').
    pred_idx = torch.argmax(new_logits[0, 1, :])
    print(f"After 'anugrah', the model predicts: '{idx_to_word[pred_idx.item()]}'")

Epoch [20/100], Loss: 1.2234
Epoch [40/100], Loss: 0.2715
Epoch [60/100], Loss: 0.0686
Epoch [80/100], Loss: 0.0316
Epoch [100/100], Loss: 0.0175

Training complete!
After 'anugrah', the model predicts: 'is'
